# 🧰 quant-kit — Vast.ai Professional Benchmark Suite

**Professional benchmarks on A100/H100 (~$2-4 total).**

Runs benchmarks that **cannot** run on Kaggle T4 (due to Gemma's 256k vocab OOM):
- 🧠 TruthfulQA, ARC Challenge, HellaSwag, Winogrande (MC loglikelihood)
- 📐 MMLU Pro (12,032 questions across 57 subjects)
- ➕ GSM8K, IFEval (generative — double-check vs Kaggle)
- 💻 HumanEval (code generation)

### Setup on Vast.ai
1. Rent: **A100 40GB** or **RTX 4090** (24GB VRAM minimum)
2. Template: `pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime`
3. Open Jupyter, upload this file
4. Set config → Run All (~3-5 hours, ~\$2-4)

> **Why not Kaggle?** Gemma 4 has 256k vocab. MC tasks require logprobs over all 256k tokens per sample → OOM on 16GB T4 once the 7GB model is loaded. A100 (40GB) handles it easily.

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
HF_TOKEN          = "hf_your_token_here"
HF_REPO           = "Dhptl/gemma-4-12b-it-GGUF"
ORIGINAL_MODEL_ID = "google/gemma-4-12b-it"
QUANT_TYPE        = "Q4_K_M"

# ── Tasks — all enabled by default on A100 ────────────────────────────
# These CANNOT run on Kaggle T4 (256k vocab OOM)
MC_TASKS = "truthfulqa_mc2,arc_challenge,hellaswag,winogrande"

# These CAN run on T4 too — set False if already done on Kaggle
RUN_GENERATIVE = True   # gsm8k,ifeval
RUN_MMLU_PRO   = True   # ~90 min
RUN_HUMANEVAL  = True   # ~30 min
# ──────────────────────────────────────────────────────────────────────

In [ ]:
# ── Install packages ───────────────────────────────────────────────────
import subprocess, sys, os

os.environ["HF_TOKEN"] = HF_TOKEN

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-cpp-python[server]",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "lm-eval[api]", "psutil",
    "datasets", "jinja2", "requests",
    "langdetect",   # for ifeval
], check=True)

r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print(f"GPU: {r.stdout.strip()}")
print("Packages ready!")

In [ ]:
# ── Download GGUF from HuggingFace ─────────────────────────────────────
from huggingface_hub import hf_hub_download, HfApi
from pathlib import Path

model_name = HF_REPO.split("/")[1]
base_name  = model_name.replace("-GGUF", "")
gguf_file  = f"{base_name}-{QUANT_TYPE}.gguf"
model_path = f"/workspace/{gguf_file}"

print(f"Downloading {gguf_file}...")
hf_hub_download(repo_id=HF_REPO, filename=gguf_file,
                local_dir="/workspace", token=HF_TOKEN)
print(f"Ready: {Path(model_path).stat().st_size/1e9:.2f} GB")

In [ ]:
# ── Start llama-cpp-python OpenAI server ───────────────────────────────
import subprocess, sys, time
import requests as req

SERVER_PORT = 8080
SERVER_URL  = f"http://localhost:{SERVER_PORT}"

print(f"Starting server on port {SERVER_PORT}...")
log_file = open("/workspace/server_log.txt", "w")

server_proc = subprocess.Popen([
    sys.executable, "-m", "llama_cpp.server",
    "--model",        model_path,
    "--n_gpu_layers", "-1",
    "--n_ctx",        "4096",
    "--port",         str(SERVER_PORT),
    "--host",         "0.0.0.0",
], stdout=log_file, stderr=subprocess.STDOUT, text=True)

server_ready = False
for i in range(90):
    time.sleep(2)
    if server_proc.poll() is not None:
        print("\n❌ Server crashed!")
        with open("/workspace/server_log.txt") as f:
            print(f.read())
        raise RuntimeError("Server crashed.")
    try:
        r = req.get(f"{SERVER_URL}/v1/models", timeout=2)
        if r.status_code == 200:
            print(f"\n✅ Server ready after {i*2}s!")
            server_ready = True
            break
    except Exception:
        pass

if not server_ready:
    server_proc.terminate()
    with open("/workspace/server_log.txt") as f:
        print("".join(f.readlines()[-20:]))
    raise RuntimeError("Server timed out.")

print(f"Server at {SERVER_URL}")

In [ ]:
# ── Run ALL benchmarks ─────────────────────────────────────────────────
import subprocess, sys, json
from pathlib import Path

eval_results = {}
results_dir  = Path("/workspace/eval_results")
results_dir.mkdir(exist_ok=True)

def run_tasks(task_str, label, timeout=21600):
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model",       "gguf",
        "--model_args",  f"base_url={SERVER_URL}",
        "--tasks",       task_str,
        "--output_path", str(results_dir),
        "--batch_size",  "1",
    ]
    subprocess.run(cmd, text=True, timeout=timeout)

# 1. MC tasks (need big VRAM — this is why we're on Vast.ai!)
run_tasks(MC_TASKS,
    "MC Tasks: TruthfulQA, ARC, HellaSwag, Winogrande (~1-2 hrs)")

# 2. Generative tasks
if RUN_GENERATIVE:
    run_tasks("gsm8k,ifeval",
        "Generative: GSM8K + IFEval (~1 hr)")

# 3. MMLU Pro
if RUN_MMLU_PRO:
    run_tasks("mmlu_pro",
        "MMLU Pro — 57 subjects, 12,032 questions (~90 min)")

# 4. HumanEval
if RUN_HUMANEVAL:
    run_tasks("humaneval",
        "HumanEval — 164 code problems (~30 min)")

# Kill server
try:
    server_proc.terminate()
    print("\nServer stopped.")
except Exception:
    pass

# Collect all results
for result_file in results_dir.glob("**/*.json"):
    if "results" in result_file.name:
        with open(result_file) as f:
            data = json.load(f)
        for task, metrics in data.get("results", {}).items():
            score = (
                metrics.get("acc_norm,none") or
                metrics.get("acc,none") or
                metrics.get("exact_match,none") or
                metrics.get("prompt_level_strict_acc,none")
            )
            if score is not None:
                eval_results[task] = round(score * 100, 2)
        break

print("\n── All Results ──")
for task, score in eval_results.items():
    print(f"  {task}: {score}%")

In [ ]:
# ── Upload results to HuggingFace ──────────────────────────────────────
import json
from huggingface_hub import HfApi

output = {
    "model":      HF_REPO,
    "quant":      QUANT_TYPE,
    "platform":   "Vast.ai A100 GPU",
    "benchmarks": eval_results,
}

result_file = f"/workspace/vastai_results_{QUANT_TYPE}.json"
with open(result_file, "w") as f:
    json.dump(output, f, indent=2)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=result_file,
    path_in_repo=f"vastai_results_{QUANT_TYPE}.json",
    repo_id=HF_REPO, repo_type="model",
    commit_message=f"Add Vast.ai professional benchmark results ({QUANT_TYPE})"
)

print(f"\n✅ Results uploaded to https://huggingface.co/{HF_REPO}")
print("\nNow run on your laptop to update the README:")
print(f"  python model_card.py --model {base_name} --original {ORIGINAL_MODEL_ID}")
print(f"  python upload.py --model {base_name}")
print()
print(json.dumps(output, indent=2))